# Практическая часть выпускной квалификационной работы

## Прогнозирование волатильности индекса МосБиржи на основе моделей ARCH/GARCH

**Выполнила:** Семенова Дарья Александровна, студентка 4 курса, группа 22.Б03  
**Направление:** 01.03.02 «Прикладная математика и информатика»  
**Университет:** Санкт-Петербургский государственный университет  
**Год:** 2026

---

В практической части рассматривается задача прогнозирования волатильности биржевого временного ряда с помощью моделей условной гетероскедастичности ARCH/GARCH. В качестве эмпирического объекта выбран индекс МосБиржи (IMOEX), поскольку он отражает агрегированную динамику российского фондового рынка и позволяет анализировать не риск отдельной компании, а общий рыночный риск.

Цель практической части — построить и сравнить несколько подходов к прогнозированию условной волатильности доходностей IMOEX, а затем проверить качество прогнозов на тестовом периоде.

Основные этапы исследования:

1. загрузка дневных данных IMOEX и переход к логарифмическим доходностям;
2. диагностика свойств ряда, важных для моделей волатильности;
3. построение модели условного среднего ARMA;
4. оценивание ARCH(5), GARCH(1,1) и GJR-GARCH(1,1,1)-t;
5. вневыборочное сравнение прогнозов волатильности;
6. проверка итоговой модели через Value-at-Risk.

## 0. Используемые библиотеки

In [1]:
# !pip install pandas numpy plotly scipy statsmodels arch requests nbformat


In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from scipy import stats
from scipy.stats import chi2, norm as scipy_norm, t as scipy_t

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.arima.model import ARIMA

from arch import arch_model

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

np.random.seed(42)

## 1. Данные и объект исследования

Данные загружаются из MOEX ISS API. Используются дневные значения индекса IMOEX за период с `2015-01-01` по `2026-04-30`.

Период до `2024-01-01` используется как обучающая выборка


In [3]:
start_date, end_date, test_start = "2015-01-01", "2026-04-30", "2024-01-01"

def load_moex_candles(secid, market, start, end, interval=24):
    all_rows, offset = [], 0
    for _ in range(200):
        url = f"https://iss.moex.com/iss/engines/stock/markets/{market}/securities/{secid}/candles.json"
        r = requests.get(url, params={"from": start, "till": end, "interval": interval, "start": offset}, timeout=30)
        r.raise_for_status()
        data = r.json()
        rows = data["candles"]["data"]
        if not rows:
            break
        all_rows.extend(rows)
        offset += len(rows)
    df = pd.DataFrame(all_rows, columns=data["candles"]["columns"])
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.sort_values("begin").reset_index(drop=True)
    for col in ["open", "close", "high", "low", "value", "volume"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

asset = load_moex_candles("IMOEX", "index", start_date, end_date)
asset["log_return"] = 100 * np.log(asset["close"]).diff()
ret = asset.set_index("begin")["log_return"].dropna().astype(float)

split_date = pd.Timestamp(test_start)
ret_train = ret[ret.index < split_date]
ret_test  = ret[ret.index >= split_date]

print(f"IMOEX: {len(asset)} наблюдений,  {asset['begin'].min().date()} — {asset['begin'].max().date()}")
print(f"Train: {len(ret_train)}; Test: {len(ret_test)}")
asset.head()

IMOEX: 2846 наблюдений,  2015-01-05 — 2026-04-30
Train: 2253; Test: 592


,open,close,high,low,value,volume,begin,end,log_return
0,"1,394.6600","1,435.6600","1,438.9100","1,390.5300","15,320,687,285.0000",0,2015-01-05,2015-01-05 23:59:59,NaN
1,"1,435.3900","1,480.7300","1,481.3500","1,430.4700","21,268,814,897.0000",0,2015-01-06,2015-01-06 23:59:59,3.0911
2,"1,482.1200","1,547.3900","1,564.8700","1,481.9300","35,505,063,516.0000",0,2015-01-08,2015-01-08 23:59:59,4.4034
3,"1,547.6100","1,515.3700","1,557.9600","1,496.1700","27,223,058,410.0000",0,2015-01-09,2015-01-09 23:59:59,-2.0910
4,"1,515.2300","1,513.2200","1,534.7400","1,503.4900","23,419,055,856.0000",0,2015-01-12,2015-01-12 23:59:59,-0.1420


### Переход к доходностям

ARCH/GARCH-модели строятся по доходностям, а не по уровню индекса. Уровень индекса обычно нестационарен, тогда как логарифмические доходности лучше подходят для моделирования финансовых временных рядов.

Используется дневная логарифмическая доходность:

$$
r_t = 100 \cdot \ln\frac{P_t}{P_{t-1}}.
$$

Множитель 100 переводит доходности в проценты и делает параметры моделей удобнее для интерпретации.


In [4]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Значение индекса IMOEX", "Логарифмическая дневная доходность IMOEX, %"))

fig.add_trace(go.Scatter(x=asset["begin"], y=asset["close"], mode="lines", name="Цена закрытия", line=dict(color="#1f77b4", width=1.5)),
              row=1, col=1)

ret_plot = ret.reset_index()
fig.add_trace(go.Scatter(x=ret_plot["begin"], y=ret_plot["log_return"], mode="lines", name="Лог-доходность", line=dict(color="#d62728", width=0.8)),
              row=2, col=1)

fig.add_vrect(x0=test_start, x1=end_date, fillcolor="LightGreen", opacity=0.18, line_width=0, annotation_text="тест", annotation_position="top left",
              row="all", col=1)

fig.update_layout(title="Исходный ряд цен и переход к доходностям", height=680, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.update_yaxes(title_text="руб.", row=1, col=1)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.show()


## 2. Предварительный анализ доходностей

Проверяем следующие свойста:

- стационарность доходностей;
- отклонение распределения от нормального и наличие тяжелых хвостов;
- автокорреляция доходностей;
- автокорреляция квадратов доходностей;
- изменение исторической волатильности во времени.

Эти проверки нужны для обоснования перехода от обычной модели среднего к модели условной дисперсии.


In [5]:
jb = stats.jarque_bera(ret)

desc = pd.DataFrame(
    {"Метрика": [
            "Среднее, %",
            "Стандартное отклонение, %",
            "Минимум, %",
            "Максимум, %",
            "Асимметрия",
            "Эксцесс / kurtosis",
            "Jarque-Bera p-value"],
        "Значение": [
            ret.mean(),
            ret.std(),
            ret.min(),
            ret.max(),
            stats.skew(ret),
            stats.kurtosis(ret, fisher=False),
            jb.pvalue]})

display(desc)


,Метрика,Значение
0,"Среднее, %",0.0217
1,"Стандартное отклонение, %",1.5230
2,"Минимум, %",-40.4674
3,"Максимум, %",18.2620
4,Асимметрия,-6.4234
5,Эксцесс / kurtosis,188.8490
6,Jarque-Bera p-value,0.0000


Для моделей волатильности особенно важны асимметрия и эксцесс. Асимметрия показывает, отличаются ли левый и правый хвосты распределения. Эксцесс выше 3 означает тяжелые хвосты, то есть более частые экстремальные наблюдения по сравнению с нормальным распределением.

Jarque-Bera используется здесь как формальная проверка нормальности. Если нормальность отвергается, это будет аргументом в пользу t-критерия в GARCH-модели.


In [6]:
x_grid = np.linspace(ret.quantile(0.001), ret.quantile(0.999), 600)
normal_pdf = stats.norm.pdf(x_grid, ret.mean(), ret.std())
osm, osr = stats.probplot(ret, dist="norm", fit=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Распределение доходностей и нормальное приближение",
    "QQ-plot относительно нормального распределения",
))

fig.add_trace(go.Histogram(x=ret, histnorm="probability density", nbinsx=120,
                           name="Доходности", marker_color="#4c78a8", opacity=0.72), row=1, col=1)
fig.add_trace(go.Scatter(x=x_grid, y=normal_pdf, mode="lines", name="Нормальное распределение",
                         line=dict(color="#f58518", width=2)), row=1, col=1)

q_min = min(np.min(osm), np.min(osr))
q_max = max(np.max(osm), np.max(osr))
fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", name="QQ-точки",
                         marker=dict(size=4, color="#54a24b", opacity=0.65)), row=1, col=2)
fig.add_trace(go.Scatter(x=[q_min, q_max], y=[q_min, q_max], mode="lines", name="Нормальная линия",
                         line=dict(color="#e45756", dash="dash")), row=1, col=2)

fig.update_layout(title="Проверка распределения доходностей", height=520, bargap=0.03,
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.update_xaxes(title_text="r_t, %", row=1, col=1)
fig.update_yaxes(title_text="Плотность", row=1, col=1)
fig.update_xaxes(title_text="Теоретические квантили", row=1, col=2)
fig.update_yaxes(title_text="Наблюдаемые квантили", row=1, col=2)
fig.show()

In [7]:
rolling_22 = ret.rolling(22).std()
rolling_63 = ret.rolling(63).std()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_22.index, y=rolling_22, mode="lines", name="22 торговых дня",
                         line=dict(color="#1f77b4", width=1.5)))
fig.add_trace(go.Scatter(x=rolling_63.index, y=rolling_63, mode="lines", name="63 торговых дня",
                         line=dict(color="#ff7f0e", width=1.5)))
fig.add_vrect(x0=test_start, x1=end_date, fillcolor="LightGreen", opacity=0.16, line_width=0,
              annotation_text="тест", annotation_position="top left")
fig.update_layout(title="Скользящая историческая волатильность IMOEX", xaxis_title="Дата",
                  yaxis_title="Стандартное отклонение дневной доходности, %", height=460,
                  hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()

Скользящее стандартное отклонение показывает, как менялась историческая волатильность. Окно 22 торговых дня соответствует примерно одному месяцу, окно 63 дня — примерно кварталу.

22-дневная историческая волатильность далее используется как наивный бенчмарк. Это важная точка сравнения: ARCH/GARCH-модель должна давать прогноз лучше простого исторического подхода, иначе усложнение модели плохо оправдано.


## 3. Статистические проверки

Используются несколько критериев, специфичных для анализа временных рядов.

**ADF-тест** проверяет стационарность ряда доходностей. Для ARCH/GARCH важно работать со стационарным рядом, поэтому проверяется именно доходность, а не уровень индекса.

**Jarque-Bera** проверяет нормальность распределения доходностей. Этот тест нужен для выбора распределения инноваций: если нормальность отвергается, нормальная GARCH-модель может плохо описывать хвостовые риски.

**Ljung-Box для $r_t$** проверяет наличие автокорреляции в самих доходностях. Если она есть, перед GARCH нужно построить модель условного среднего.

**Ljung-Box для $r_t^2$** и **ARCH-LM** проверяют наличие ARCH-эффекта. Это означает, что величина текущих колебаний зависит от прошлых шоков, то есть волатильность предсказуема во времени.


In [8]:
def p_decision(p_value, reject_text, keep_text, alpha=0.05):
    return reject_text if p_value < alpha else keep_text

adf_stat, adf_p, *_ = adfuller(ret)
lb_ret = acorr_ljungbox(ret, lags=[10], return_df=True).iloc[0]
lb_sq = acorr_ljungbox(ret**2, lags=[10], return_df=True).iloc[0]
lm_stat, lm_p, _, _ = het_arch(ret, nlags=10)

tests = pd.DataFrame([
    {"Проверка": "ADF: единичный корень", "Статистика": adf_stat, "p-value": adf_p,
     "Интерпретация": p_decision(adf_p, "Доходности стационарны", "Стационарность не подтверждена")},
    {"Проверка": "Jarque-Bera: нормальность", "Статистика": jb.statistic, "p-value": jb.pvalue,
     "Интерпретация": p_decision(jb.pvalue, "Нормальность отвергается, нужны тяжелые хвосты", "Нет оснований отвергать нормальность")},
    {"Проверка": "Ljung-Box для r_t, lag=10", "Статистика": lb_ret["lb_stat"], "p-value": lb_ret["lb_pvalue"],
     "Интерпретация": p_decision(lb_ret["lb_pvalue"], "Есть автокорреляция, нужна модель среднего", "Автокорреляция незначима")},
    {"Проверка": "Ljung-Box для r_t^2, lag=10", "Статистика": lb_sq["lb_stat"], "p-value": lb_sq["lb_pvalue"],
     "Интерпретация": p_decision(lb_sq["lb_pvalue"], "Есть ARCH-эффект, нужна модель волатильности", "ARCH-эффект не подтвержден")},
    {"Проверка": "ARCH-LM для r_t, lag=10", "Статистика": lm_stat, "p-value": lm_p,
     "Интерпретация": p_decision(lm_p, "Условная гетероскедастичность подтверждена", "ARCH-эффект не подтвержден")},
])

display(tests)

,Проверка,Статистика,p-value,Интерпретация
0,ADF: единичный корень,-19.4353,0.0000,Доходности стационарны
1,Jarque-Bera: нормальность,"4,113,973.8732",0.0000,"Нормальность отвергается, нужны тяжелые хвосты"
2,"Ljung-Box для r_t, lag=10",30.4551,0.0007,"Есть автокорреляция, нужна модель среднего"
3,"Ljung-Box для r_t^2, lag=10",139.0840,0.0000,"Есть ARCH-эффект, нужна модель волатильности"
4,"ARCH-LM для r_t, lag=10",118.8992,0.0000,Условная гетероскедастичность подтверждена


In [9]:
def add_corr_plot(fig, series, row, col, title, nlags=30, kind="acf"):
    values = acf(series, nlags=nlags, fft=True) if kind == "acf" else pacf(series, nlags=nlags, method="ywm")
    conf = 1.96 / np.sqrt(series.dropna().shape[0])
    fig.add_hrect(y0=-conf, y1=conf, fillcolor="#9ecae1", opacity=0.25, line_width=0, row=row, col=col)
    fig.add_trace(go.Bar(x=np.arange(len(values)), y=values, name=title,
                         marker_color="#4c78a8", showlegend=False), row=row, col=col)
    fig.update_xaxes(title_text="Лаг", row=row, col=col)
    fig.update_yaxes(title_text=kind.upper(), range=[-0.25, 0.45], row=row, col=col)

fig = make_subplots(rows=2, cols=2, subplot_titles=(
    "ACF доходностей", "PACF доходностей",
    "ACF квадратов доходностей", "PACF квадратов доходностей",
))

add_corr_plot(fig, ret, 1, 1, "ACF r_t", kind="acf")
add_corr_plot(fig, ret, 1, 2, "PACF r_t", kind="pacf")
add_corr_plot(fig, ret**2, 2, 1, "ACF r_t^2", kind="acf")
add_corr_plot(fig, ret**2, 2, 2, "PACF r_t^2", kind="pacf")

fig.update_layout(title="ACF/PACF: проверка памяти в доходностях и волатильности", height=760, bargap=0.18)
fig.show()

### ACF/PACF и ARCH-эффект

ACF показывает автокорреляции на разных лагах, PACF — прямую связь с лагом после исключения промежуточных лагов. На графиках синяя область соответствует приближенной зоне статистической незначимости.

Для доходностей $r_t$ значимые лаги указывают на необходимость модели условного среднего. Для квадратов доходностей $r_t^2$ значимые лаги важнее: они показывают зависимость в масштабе колебаний, то есть кластеризацию волатильности.

Если квадраты доходностей автокоррелированы, то предположение о постоянной дисперсии ошибок не подходит, и использование ARCH/GARCH становится обоснованным.


## 4. Модель условного среднего ARMA

Перед моделированием волатильности из доходностей удаляется линейная зависимость. Для этого строится ARMA(p, q): AR-часть учитывает зависимость от прошлых доходностей, MA-часть — зависимость от прошлых ошибок.

Порядок модели выбирается по AIC на обучающей выборке среди вариантов с $p, q \le 3$. AIC используется как критерий баланса между качеством подгонки и числом параметров.

После выбора ARMA дальнейшие модели волатильности строятся уже по остаткам этой модели.


In [10]:
arma_results = []
for p in range(0, 4):
    for q in range(0, 4):
        if p == 0 and q == 0:
            continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model = ARIMA(ret_train, order=(p, 0, q)).fit()
            arma_results.append({"p": p, "q": q, "ARMA": f"ARMA({p},{q})",
                                  "AIC": model.aic, "BIC": model.bic, "LogLik": model.llf})
        except Exception:
            pass

arma_table = pd.DataFrame(arma_results).sort_values("AIC").reset_index(drop=True)
display(arma_table[["ARMA", "AIC", "BIC", "LogLik"]].head(8))

P = int(arma_table.loc[0, "p"])
Q = int(arma_table.loc[0, "q"])
print(f"Выбранная модель условного среднего: ARMA({P},{Q}) по минимальному AIC.")

,ARMA,AIC,BIC,LogLik
0,"ARMA(2,3)","8,401.3251","8,441.3653","-4,193.6626"
1,"ARMA(3,2)","8,401.3307","8,441.3709","-4,193.6654"
2,"ARMA(3,3)","8,403.3421","8,449.1023","-4,193.6711"
3,"ARMA(2,2)","8,404.6983","8,439.0185","-4,196.3492"
4,"ARMA(1,0)","8,409.3526","8,426.5126","-4,201.6763"
5,"ARMA(0,1)","8,410.0340","8,427.1941","-4,202.0170"
6,"ARMA(0,2)","8,410.5741","8,433.4541","-4,201.2870"
7,"ARMA(1,2)","8,410.6544","8,439.2545","-4,200.3272"


Выбранная модель условного среднего: ARMA(2,3) по минимальному AIC.


In [11]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mean_model = ARIMA(ret_train, order=(P, 0, Q)).fit()

eps_train = mean_model.resid.dropna()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    eps_full = mean_model.apply(ret).resid.dropna()
mu_full = ret.loc[eps_full.index] - eps_full

eps_train = eps_full.loc[eps_full.index < split_date].dropna()
eps_test = eps_full.loc[eps_full.index >= split_date].dropna()
mu_test = mu_full.loc[eps_test.index]
ret_test_aligned = ret.loc[eps_test.index]

lb_eps = acorr_ljungbox(eps_train, lags=[10], return_df=True).iloc[0]
lm_eps_stat, lm_eps_p, _, _ = het_arch(eps_train, nlags=10)

residual_tests = pd.DataFrame([
    {"Проверка": "Ljung-Box для остатков ARMA, lag=10", "Статистика": lb_eps["lb_stat"],
     "p-value": lb_eps["lb_pvalue"],
     "Интерпретация": p_decision(lb_eps["lb_pvalue"], "В остатках сохраняется автокорреляция",
                                 "Линейная автокорреляция существенно снижена")},
    {"Проверка": "ARCH-LM для остатков ARMA, lag=10", "Статистика": lm_eps_stat,
     "p-value": lm_eps_p,
     "Интерпретация": p_decision(lm_eps_p, "ARCH-эффект сохраняется, строим GARCH",
                                 "ARCH-эффект не подтвержден")},
])

display(residual_tests)

,Проверка,Статистика,p-value,Интерпретация
0,"Ljung-Box для остатков ARMA, lag=10",7.7347,0.6547,Линейная автокорреляция существенно снижена
1,"ARCH-LM для остатков ARMA, lag=10",64.8228,0.0000,"ARCH-эффект сохраняется, строим GARCH"


### Диагностика остатков ARMA

После оценки ARMA проверяется, осталась ли в остатках линейная автокорреляция. Если она устранена, то модель среднего можно считать достаточной для дальнейшего анализа.

Далее проверяется ARCH-LM на остатках. Если он остается значимым, это означает, что структура среднего уже учтена, но условная дисперсия остатков все еще меняется во времени.


In [12]:
eps_acf = acf(eps_train, nlags=30, fft=True)
conf_eps = 1.96 / np.sqrt(len(eps_train))

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Остатки ARMA({P},{Q}) на обучающей выборке", "ACF остатков ARMA",
))

fig.add_trace(go.Scatter(x=eps_train.index, y=eps_train, mode="lines", name="Остатки",
                         line=dict(color="#2ca02c", width=0.8)), row=1, col=1)
fig.add_hline(y=0, line_width=1, line_dash="dash", line_color="black", row=1, col=1)

fig.add_hrect(y0=-conf_eps, y1=conf_eps, fillcolor="#9ecae1", opacity=0.25, line_width=0, row=1, col=2)
fig.add_trace(go.Bar(x=np.arange(len(eps_acf)), y=eps_acf, name="ACF",
                     marker_color="#4c78a8", showlegend=False), row=1, col=2)

fig.update_layout(title="Диагностика модели условного среднего", height=460, hovermode="x unified")
fig.update_yaxes(title_text="остаток, п.п.", row=1, col=1)
fig.update_xaxes(title_text="Дата", row=1, col=1)
fig.update_xaxes(title_text="Лаг", row=1, col=2)
fig.update_yaxes(title_text="ACF", row=1, col=2)
fig.show()

По результатам диагностики линейная автокорреляция в остатках ARMA существенно снижена. При этом ARCH-LM остается значимым, поэтому в остатках сохраняется условная гетероскедастичность.

Именно эту оставшуюся зависимость в дисперсии далее моделируют ARCH/GARCH-модели.


## 5. Модели условной волатильности

Сравниваются четыре подхода.

| Подход | Роль в работе |
|---|---|
| Наивный бенчмарк | 22-дневное скользящее стандартное отклонение. Нужен как простой ориентир. |
| ARCH(5) | Дисперсия зависит от квадратов последних пяти шоков. Это базовая модель условной гетероскедастичности. |
| GARCH(1,1) | Дисперсия зависит от прошлого шока и прошлого значения самой дисперсии. Это стандартная модель финансовой волатильности. |
| GJR-GARCH(1,1,1)-t | Расширяет GARCH за счет асимметрии отрицательных шоков и t-распределения |

GJR-GARCH-t включается потому, что предварительный анализ показал тяжелые хвосты доходностей, а финансовые рынки часто реагируют на отрицательные шоки сильнее, чем на положительные.


In [13]:
arch5 = arch_model(eps_train, mean="Zero", vol="ARCH", p=5, dist="normal").fit(disp="off")
garch = arch_model(eps_train, mean="Zero", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
gjr = arch_model(eps_train, mean="Zero", vol="GARCH", p=1, o=1, q=1, dist="t").fit(disp="off")

print("Модели волатильности оценены на остатках ARMA обучающей выборки.")


Модели волатильности оценены на остатках ARMA обучающей выборки.


### Параметры моделей


In [14]:
def confint_table(result, model_name):
    params = result.params
    stderr = result.std_err
    rows = []
    for name, value in params.items():
        se = stderr[name]
        lower, upper = value - 1.96 * se, value + 1.96 * se
        rows.append({"Модель": model_name, "Параметр": name, "Оценка": value, "SE": se,
                     "95% нижняя": lower, "95% верхняя": upper,
                     "Статус": "значим" if not (lower <= 0 <= upper) else "не значим"})
    return pd.DataFrame(rows)

param_table = pd.concat([
    confint_table(arch5, "ARCH(5)"),
    confint_table(garch, "GARCH(1,1)"),
    confint_table(gjr, "GJR-GARCH-t"),
], ignore_index=True)

display(param_table)

,Модель,Параметр,Оценка,SE,95% нижняя,95% верхняя,Статус
0,ARCH(5),omega,0.5959,0.1080,0.3841,0.8076,значим
1,ARCH(5),alpha[1],0.1679,0.0616,0.0473,0.2886,значим
2,ARCH(5),alpha[2],0.2856,0.0984,0.0926,0.4785,значим
3,ARCH(5),alpha[3],0.0863,0.0413,0.0054,0.1672,значим
4,ARCH(5),alpha[4],0.0920,0.0303,0.0326,0.1515,значим
5,ARCH(5),alpha[5],0.0684,0.0263,0.0168,0.1199,значим
6,"GARCH(1,1)",omega,0.0306,0.0180,-0.0047,0.0659,не значим
7,"GARCH(1,1)",alpha[1],0.1287,0.0485,0.0337,0.2237,значим
8,"GARCH(1,1)",beta[1],0.8647,0.0439,0.7787,0.9507,значим
9,GJR-GARCH-t,omega,0.0425,0.0098,0.0234,0.0617,значим


Интерпретация параметров следующая.

- $\omega$ — базовый уровень условной дисперсии.
- $\alpha$ — реакция волатильности на новый шок.
- $\beta$ — перенос прошлой дисперсии в текущий прогноз.
- $\gamma$ — дополнительный эффект отрицательного шока в GJR-GARCH. Положительный значимый $\gamma$ означает, что падения рынка сильнее увеличивают будущую волатильность.
- $\nu$ — число степеней свободы t-распределения. Чем меньше $\nu$, тем тяжелее хвосты.

Для IMOEX значимость $\gamma$ указывает на асимметричную реакцию волатильности, а значимость $\nu$ подтверждает необходимость распределения с тяжелыми хвостами.


### Персистентность волатильности

Персистентность показывает, как долго эффект шока сохраняется в условной дисперсии. Для GARCH(1,1) она оценивается как $\alpha + \beta$, для GJR-GARCH — как $\alpha + \beta + \gamma/2$.

Полупериод шока переводит эту величину в более понятную форму: сколько торговых дней требуется, чтобы влияние шока на прогнозную дисперсию сократилось примерно вдвое.


In [15]:
def model_persistence(name, result):
    params = result.params
    if name == "ARCH(5)":
        return sum(v for k, v in params.items() if k.startswith("alpha"))
    if name == "GARCH(1,1)":
        return params.get("alpha[1]", np.nan) + params.get("beta[1]", np.nan)
    if name == "GJR-GARCH-t":
        return params.get("alpha[1]", np.nan) + params.get("beta[1]", np.nan) + 0.5 * params.get("gamma[1]", 0)
    return np.nan

def half_life(persistence):
    if persistence <= 0 or persistence >= 1:
        return np.nan
    return np.log(0.5) / np.log(persistence)

comparison = []
for name, result in [("ARCH(5)", arch5), ("GARCH(1,1)", garch), ("GJR-GARCH-t", gjr)]:
    p = model_persistence(name, result)
    comparison.append({"Модель": name, "AIC": result.aic, "BIC": result.bic,
                       "LogLik": result.loglikelihood, "Персистентность": p,
                       "Полупериод шока, дней": half_life(p)})

insample_table = pd.DataFrame(comparison).sort_values("AIC").reset_index(drop=True)
display(insample_table)

delta_garch_arch = arch5.aic - garch.aic
delta_gjr_garch = garch.aic - gjr.aic
print(f"ΔAIC ARCH(5) - GARCH(1,1): {delta_garch_arch:.1f}")
print(f"ΔAIC GARCH(1,1) - GJR-GARCH-t: {delta_gjr_garch:.1f}")

,Модель,AIC,BIC,LogLik,Персистентность,"Полупериод шока, дней"
0,GJR-GARCH-t,"6,610.8652","6,639.4653","-3,300.4326",0.9647,19.2779
1,"GARCH(1,1)","6,881.0049","6,898.1649","-3,437.5024",0.9934,104.4054
2,ARCH(5),"7,001.8233","7,036.1434","-3,494.9117",0.7002,1.9451


ΔAIC ARCH(5) - GARCH(1,1): 120.8
ΔAIC GARCH(1,1) - GJR-GARCH-t: 270.1


По AIC и BIC лучшей внутри выборки является GJR-GARCH-t. Это означает, что добавление асимметрии и t-инноваций существенно улучшает описание данных по сравнению с ARCH(5) и GARCH(1,1).

Однако внутривыборочные критерии не отвечают на вопрос о прогнозной точности. Поэтому окончательный выбор модели должен учитывать вневыборочные метрики и тесты сравнения прогнозов.


### Оцененная условная волатильность

График показывает оценку изменяющегося во времени уровня риска. В отличие от скользящего стандартного отклонения, GARCH-модели используют параметрическую зависимость дисперсии от прошлых шоков и прошлой дисперсии.

Сравнение GARCH(1,1) и GJR-GARCH-t показывает, насколько учет асимметрии и тяжелых хвостов влияет на оценку волатильности в стрессовые периоды.


In [16]:
def fixed_volatility_result(eps_series, fitted_result, vol, p, q=0, o=0, dist="normal"):
    model = arch_model(eps_series, mean="Zero", vol=vol, p=p, o=o, q=q, dist=dist)
    return model.fix(fitted_result.params)

fixed_arch = fixed_volatility_result(eps_full, arch5, "ARCH", p=5, q=0, o=0, dist="normal")
fixed_garch = fixed_volatility_result(eps_full, garch, "GARCH", p=1, q=1, o=0, dist="normal")
fixed_gjr = fixed_volatility_result(eps_full, gjr, "GARCH", p=1, q=1, o=1, dist="t")

sigma_arch = fixed_arch.conditional_volatility
sigma_garch = fixed_garch.conditional_volatility
sigma_gjr = fixed_gjr.conditional_volatility

fig = go.Figure()
fig.add_trace(go.Scatter(x=sigma_garch.index, y=sigma_garch, mode="lines", name="GARCH(1,1)",
                         line=dict(color="#1f77b4", width=1.2)))
fig.add_trace(go.Scatter(x=sigma_gjr.index, y=sigma_gjr, mode="lines", name="GJR-GARCH-t",
                         line=dict(color="#d62728", width=1.2)))
fig.add_vrect(x0=test_start, x1=end_date, fillcolor="LightGreen", opacity=0.16, line_width=0,
              annotation_text="тест", annotation_position="top left")
fig.update_layout(title="Оцененная условная волатильность по GARCH-моделям", xaxis_title="Дата",
                  yaxis_title="σ_t, процентных пункта", height=480, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()

Обе модели выделяют основные периоды повышенной волатильности. GARCH(1,1) имеет более высокую персистентность, поэтому дольше сохраняет повышенный прогноз после шока. GJR-GARCH-t учитывает знак шока и тяжелые хвосты, поэтому его реакция на отдельные кризисные движения отличается.

Этот график показывает поведение моделей на истории, но качество прогноза проверяется отдельно на тестовой выборке.


## 6. Прогнозирование волатильности

Для проверки прогноза используется тестовый период после `2024-01-01`. Параметры моделей оцениваются на обучающей выборке и затем применяются к тестовому периоду.

Истинная условная волатильность не наблюдается, поэтому в качестве прокси используется квадрат остатка модели среднего $\hat\varepsilon_t^2$. Поскольку такой прокси шумный, кроме MSE и MAE используется QLIKE — функция потерь, специально применяемая для сравнения прогнозов дисперсии.


In [17]:
sigma2_true = eps_test**2

f_naive = eps_full.rolling(22).std().shift(1).loc[eps_test.index] ** 2
f_arch = sigma_arch.loc[eps_test.index] ** 2
f_garch = sigma_garch.loc[eps_test.index] ** 2
f_gjr = sigma_gjr.loc[eps_test.index] ** 2

def qlike(y_true, y_pred, eps=1e-12):
    return np.mean(np.log(y_pred + eps) + y_true / (y_pred + eps))

def evaluate_forecast(name, pred):
    mask = pred.notna() & sigma2_true.notna()
    target, forecast = sigma2_true.loc[mask], pred.loc[mask]
    return {"Модель": name, "MSE": np.mean((target - forecast) ** 2),
            "MAE": np.mean(np.abs(target - forecast)), "QLIKE": qlike(target, forecast), "N": mask.sum()}

forecast_metrics = pd.DataFrame([
    evaluate_forecast("Naive: rolling std 22d", f_naive),
    evaluate_forecast("ARCH(5)", f_arch),
    evaluate_forecast("GARCH(1,1)", f_garch),
    evaluate_forecast("GJR-GARCH-t", f_gjr),
]).sort_values("QLIKE")

display(forecast_metrics)
print(f"Лучшая модель по QLIKE: {forecast_metrics.iloc[0]['Модель']}")
print(f"Лучшая модель по MSE: {forecast_metrics.sort_values('MSE').iloc[0]['Модель']}")

,Модель,MSE,MAE,QLIKE,N
3,GJR-GARCH-t,19.8846,1.8917,1.3857,592
2,"GARCH(1,1)",20.6734,2.0511,1.3876,592
1,ARCH(5),21.6462,2.0490,1.4759,592
0,Naive: rolling std 22d,20.9707,2.0287,1.4940,592


Лучшая модель по QLIKE: GJR-GARCH-t
Лучшая модель по MSE: GJR-GARCH-t


In [18]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=eps_test.index, y=np.abs(eps_test), mode="lines", name="|остаток ARMA|",
                         line=dict(color="rgba(120,120,120,0.55)", width=0.8)))
fig.add_trace(go.Scatter(x=f_naive.index, y=np.sqrt(f_naive), mode="lines", name="Naive 22d",
                         line=dict(color="#9467bd", width=1.2)))
fig.add_trace(go.Scatter(x=f_garch.index, y=np.sqrt(f_garch), mode="lines", name="GARCH(1,1)",
                         line=dict(color="#1f77b4", width=1.3)))
fig.add_trace(go.Scatter(x=f_gjr.index, y=np.sqrt(f_gjr), mode="lines", name="GJR-GARCH-t",
                         line=dict(color="#d62728", width=1.4)))
fig.update_layout(title="Walk-forward прогноз волатильности на тестовом периоде", xaxis_title="Дата",
                  yaxis_title="σ_t, процентных пункта", height=520, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()

По вневыборочным метрикам GJR-GARCH-t имеет наименьшие QLIKE и MSE. Наивный бенчмарк уступает параметрическим моделям по QLIKE, то есть для IMOEX моделирование условной дисперсии дает прирост качества относительно простой исторической волатильности.

Чтобы понять, является ли разница статистически значимой, далее используется тест Diebold-Mariano.


## 7. Тест Diebold-Mariano

Diebold-Mariano используется для сравнения прогнозной точности двух моделей. В данной работе он применяется к ряду разностей QLIKE-потерь.


In [19]:
def dm_test(y_true, pred_1, pred_2):
    eps = 1e-12
    d = (np.log(pred_1 + eps) + y_true / (pred_1 + eps)
         - np.log(pred_2 + eps) - y_true / (pred_2 + eps)).dropna()
    stat = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    return stat, 2 * (1 - scipy_norm.cdf(abs(stat))), len(d)

dm_rows = []
for first_name, first_pred, second_name, second_pred in [
    ("GJR-GARCH-t", f_gjr, "Naive 22d", f_naive),
    ("GARCH(1,1)", f_garch, "Naive 22d", f_naive),
    ("GJR-GARCH-t", f_gjr, "GARCH(1,1)", f_garch),
    ("GJR-GARCH-t", f_gjr, "ARCH(5)", f_arch),
]:
    stat, p_value, n_obs = dm_test(sigma2_true, first_pred, second_pred)
    vyvod = ("первая лучше статистически значимо" if stat < 0 and p_value < 0.05
             else "разница статистически незначима" if p_value >= 0.05
             else "первая хуже статистически значимо")
    dm_rows.append({"Сравнение": f"{first_name} vs {second_name}",
                    "DM": stat, "p-value": p_value, "N": n_obs, "Вывод": vyvod})

dm_table = pd.DataFrame(dm_rows)
display(dm_table)

,Сравнение,DM,p-value,N,Вывод
0,GJR-GARCH-t vs Naive 22d,-2.1185,0.0341,592,первая лучше статистически значимо
1,"GARCH(1,1) vs Naive 22d",-2.2034,0.0276,592,первая лучше статистически значимо
2,"GJR-GARCH-t vs GARCH(1,1)",-0.1574,0.8749,592,разница статистически незначима
3,GJR-GARCH-t vs ARCH(5),-3.5544,0.0004,592,первая лучше статистически значимо


GJR-GARCH-t статистически значимо превосходит наивный 22-дневный бенчмарк. Это подтверждает, что для IMOEX параметрическая модель волатильности дает полезный прогнозный выигрыш относительно простой исторической оценки.

При этом различие между GJR-GARCH-t и GARCH(1,1) статистически незначимо. То есть расширенная модель лучше описывает данные внутри выборки, но на тестовом периоде ее преимущество над обычной GARCH(1,1) не доказано.


## 8. Value-at-Risk и бэктестинг

Value-at-Risk используется как прикладная проверка прогноза волатильности. Для уровня значимости $\alpha$ дневной VaR задается как нижний квантиль условного распределения доходности:

$$
VaR_t(\alpha) = \hat\mu_t + q_\alpha \hat\sigma_t.
$$

Так как итоговая модель использует t-инновации, квантиль $q_\alpha$ берется из стандартизированного t-распределения. Затем проверяется, как часто фактическая доходность оказывается ниже VaR-линии.


In [20]:
nu = gjr.params["nu"]
q_95 = scipy_t.ppf(0.05, df=nu) * np.sqrt((nu - 2) / nu)
q_99 = scipy_t.ppf(0.01, df=nu) * np.sqrt((nu - 2) / nu)

var_95 = mu_test + q_95 * np.sqrt(f_gjr)
var_99 = mu_test + q_99 * np.sqrt(f_gjr)

def kupiec_pof_test(exceptions, n, alpha):
    x = int(exceptions)
    if x == 0:
        log_l_u = n * np.log(1 - 1e-12)
    elif x == n:
        log_l_u = n * np.log(1 - 1e-12)
    else:
        pi_hat = x / n
        log_l_u = (n - x) * np.log(1 - pi_hat) + x * np.log(pi_hat)
    log_l_r = (n - x) * np.log(1 - alpha) + x * np.log(alpha)
    lr_stat = -2 * (log_l_r - log_l_u)
    return lr_stat, 1 - chi2.cdf(lr_stat, df=1)

var_rows = []
for label, alpha, var_series in [("VaR 95%", 0.05, var_95), ("VaR 99%", 0.01, var_99)]:
    aligned = pd.concat([ret_test_aligned.rename("actual"), var_series.rename("var")], axis=1).dropna()
    exceptions = (aligned["actual"] < aligned["var"]).sum()
    n = len(aligned)
    lr_stat, kupiec_p = kupiec_pof_test(exceptions, n, alpha)
    if kupiec_p >= 0.05:
        vyvod = "покрытие не отвергается"
    elif exceptions / n < alpha:
        vyvod = "покрытие отвергается; модель консервативна"
    else:
        vyvod = "покрытие отвергается; модель занижает риск"
    var_rows.append({"Уровень": label, "Ожидаемая доля нарушений": alpha,
                     "Фактические нарушения": exceptions, "Наблюдений": n,
                     "Фактическая доля": exceptions / n,
                     "Kupiec LR": lr_stat, "Kupiec p-value": kupiec_p, "Вывод": vyvod})

var_table = pd.DataFrame(var_rows)
display(var_table)

,Уровень,Ожидаемая доля нарушений,Фактические нарушения,Наблюдений,Фактическая доля,Kupiec LR,Kupiec p-value,Вывод
0,VaR 95%,0.0500,50,592,0.0845,12.3739,0.0004,покрытие отвергается; модель занижает риск
1,VaR 99%,0.0100,6,592,0.0101,0.0011,0.9737,покрытие не отвергается


Для формальной проверки частоты нарушений используется тест Купика. Он сравнивает фактическую долю нарушений VaR с теоретически ожидаемой долей.

Для VaR 99% фактическая доля нарушений близка к 1%, поэтому модель хорошо описывает самый дальний левый хвост распределения. Для VaR 95% нарушений больше ожидаемого, значит на этом уровне модель занижает риск умеренных отрицательных движений.


In [21]:
violations_95 = ret_test_aligned[ret_test_aligned < var_95]
violations_99 = ret_test_aligned[ret_test_aligned < var_99]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ret_test_aligned.index, y=ret_test_aligned, mode="lines",
                         name="Фактическая доходность", line=dict(color="#4c78a8", width=0.9)))
fig.add_trace(go.Scatter(x=var_95.index, y=var_95, mode="lines", name="VaR 95% (GJR-GARCH-t)",
                         line=dict(color="#e45756", width=1.4)))
fig.add_trace(go.Scatter(x=var_99.index, y=var_99, mode="lines", name="VaR 99% (GJR-GARCH-t)",
                         line=dict(color="#8b0000", width=1.4, dash="dash")))
fig.add_trace(go.Scatter(x=violations_95.index, y=violations_95, mode="markers",
                         name=f"Нарушения VaR 95%: {len(violations_95)}",
                         marker=dict(color="#d62728", size=7)))
fig.add_trace(go.Scatter(x=violations_99.index, y=violations_99, mode="markers",
                         name=f"Нарушения VaR 99%: {len(violations_99)}",
                         marker=dict(color="#000000", size=8, symbol="x")))
fig.update_layout(title="Бэктестинг VaR на тестовом периоде", xaxis_title="Дата",
                  yaxis_title="Доходность, %", height=540, hovermode="x unified",
                  legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig.show()

На графике фактические доходности сравниваются с линиями VaR 95% и VaR 99%. Нарушение VaR — это наблюдение ниже соответствующей линии.

Видно, что VaR 99% нарушается примерно с ожидаемой частотой, а для VaR 95% нарушений слишком много. Это согласуется с результатами теста Купика.


## 9. Итоговые выводы практической части

1. Доходности IMOEX стационарны, имеют тяжелые хвосты и демонстрируют ARCH-эффект. Поэтому использование моделей условной волатильности методологически обосновано.

2. Модель условного среднего выбрана отдельно с помощью ARMA. После учета линейной зависимости в остатках сохраняется ARCH-эффект, поэтому дальнейшее моделирование дисперсии имеет смысл.

3. По AIC/BIC лучшей внутри выборки является GJR-GARCH-t. Ее преимущество связано с учетом асимметрии и тяжелых хвостов, которые подтверждаются диагностикой данных.

4. На тестовом периоде GJR-GARCH-t имеет лучшие значения QLIKE и MSE и статистически значимо превосходит наивную 22-дневную историческую волатильность по тесту Diebold-Mariano.

5. Разница между GJR-GARCH-t и GARCH(1,1) на тестовом периоде статистически незначима. Поэтому GJR-GARCH-t можно считать наиболее содержательной моделью, но не единственной практически конкурентоспособной спецификацией.

6. VaR 99% работает корректно: фактическая доля нарушений близка к ожидаемой. VaR 95% занижает риск, поэтому на этом уровне результаты требуют осторожной интерпретации.

**Общий вывод:** для индекса IMOEX модели ARCH/GARCH дают содержательный и практически полезный прогноз волатильности. GJR-GARCH-t является наиболее полной спецификацией среди рассмотренных моделей, но ее преимущество нужно оценивать не только по AIC/BIC, а также по вневыборочным метрикам, DM-тесту и VaR-бэктестингу.
